# 06c · Is the circuit specific to refusal? Small-set ablation vs random sets

Whole-component ablation of k = 20 (06) breaks the model for circuit and random sets alike. Here the top-k components (k = 1, 3, 5, 10) are mean-ablated and compared with 20 random sets of the same type mix, using forward passes only: the refusal logit difference on harmful prompts (should fall), the same on harmless twins (should not move) and the next-token loss on harmless prompts (coherence).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · outputs of **03 matched**, **04**, **06 matched** · `sp-models` |
| **Reads** | `pairs.jsonl`, `token_sets.json`, `probe_bundle/` |
| **Writes** (`/kaggle/working/`) | `specificity/specificity.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
import random
import numpy as np, pandas as pd, torch

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
KS = [1, 3, 5, 10] if not sp.SMOKE else [1, 2]
N_RANDOM = 20 if not sp.SMOKE else 2
SEED = 0
OUT = sp.out_dir("specificity")
pairs = sp.load_jsonl(sp.find_input("pairs.jsonl"))
val = [p for p in pairs if p["split"] == "val"]
calib = [p for p in pairs if p["split"] == "calib"]
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
bundle, _ = sp.load_bundle(sp.find_input("probe_bundle.json").parent)
ranked = [c["id"] for c in bundle["components"]]          # best first
model, tok = sp.load_model(MODEL, dtype=torch.float16)
means = sp.component_means(model, tok, [p["harmless"] for p in calib])
harm, twin = [p["harmful"] for p in val], [p["harmless"] for p in val]

copied CohereLabs__tiny-aya-global to local disk in 0.5 min


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
@torch.no_grad()
def harmless_nll(prompts):
    """Mean next-token loss of the model's own (unablated) greedy answers: rises when ablation breaks text."""
    return float(np.mean(losses(prompts)))

@torch.no_grad()
def losses(prompts):
    out = []
    for p, a in zip(prompts, answers):
        enc = sp.encode(tok, sp.format_chat(tok, [p]) , model.device)
        full = sp.encode(tok, [sp.format_chat(tok, [p])[0] + a], model.device)
        n = enc["input_ids"].shape[1]
        logits = model(**full).logits[0, n - 1:-1].float()
        tgt = full["input_ids"][0, n:]
        out.append(torch.nn.functional.cross_entropy(logits, tgt).item())
    return out

answers = sp.generate(model, tok, twin[:20], 48, desc="reference answers")

def measure(handles=None):
    with sp.hooked(handles or []):
        return {"harm_ld": float(sp.refusal_logit_diff(model, tok, harm, token_sets).mean()),
                "twin_ld": float(sp.refusal_logit_diff(model, tok, twin, token_sets).mean()),
                "nll": harmless_nll(twin[:20])}

base = measure()
print("no ablation:", base)
rng = random.Random(SEED)
rows = []
for k in KS:
    circ = measure(sp.ablate_components(model, ranked[:k], means))
    rows.append({"k": k, "set": "circuit", **circ})
    for i in range(N_RANDOM):
        rnd = sp.random_components_like(ranked[:k], model, rng)
        rows.append({"k": k, "set": "random", "seed": i, **measure(sp.ablate_components(model, rnd, means))})
    r = [x for x in rows if x["k"] == k and x["set"] == "random"]
    rank = sum(x["harm_ld"] <= circ["harm_ld"] for x in r) / len(r)   # share of random sets at least as strong
    print(f"k={k}: circuit harm_ld {circ['harm_ld']:.2f} vs random {np.mean([x['harm_ld'] for x in r]):.2f} "
          f"± {np.std([x['harm_ld'] for x in r]):.2f} (p≈{rank:.2f}); twin_ld {circ['twin_ld']:.2f}; nll {circ['nll']:.2f}")
df = pd.DataFrame(rows)
summary = (df.groupby(["k", "set"])[["harm_ld", "twin_ld", "nll"]].agg(["mean", "std"]).round(3))
sp.save_json({"base": base, "rows": rows, "n_random": N_RANDOM, "provenance": sp.provenance(SEED)},
             OUT / "specificity.json")
summary

reference answers:   0%|          | 0/3 [00:00<?, ?it/s]

no ablation: {'harm_ld': 5.898582935333252, 'twin_ld': -3.6981735229492188, 'nll': 0.35657466351985934}


k=1: circuit harm_ld 5.29 vs random 5.50 ± 0.60 (p≈0.30); twin_ld -3.94; nll 0.53


k=3: circuit harm_ld 3.65 vs random 3.93 ± 2.97 (p≈0.10); twin_ld -3.80; nll 0.90


k=5: circuit harm_ld 1.18 vs random 2.41 ± 2.26 (p≈0.20); twin_ld -3.65; nll 1.51


k=10: circuit harm_ld -2.48 vs random -1.44 ± 1.99 (p≈0.20); twin_ld -3.99; nll 3.37


harm_ld        twin_ld           nll       
              mean    std    mean    std   mean    std
k  set                                                
1  circuit   5.288    NaN  -3.939    NaN  0.528    NaN
   random    5.496  0.613  -3.579  0.294  0.464  0.045
3  circuit   3.646    NaN  -3.804    NaN  0.905    NaN
   random    3.932  3.049  -4.430  1.418  2.047  3.722
5  circuit   1.180    NaN  -3.647    NaN  1.514    NaN
   random    2.410  2.322  -3.408  0.674  2.304  3.544
10 circuit  -2.481    NaN  -3.993    NaN  3.374    NaN
   random   -1.442  2.039  -3.561  2.772  5.565  4.924